In [ ]:
import sys, os, cv2, json, numpy as np, torch
from pathlib import Path
from cotracker.utils.visualizer import Visualizer

# ── 配置 ──────────────────────────────────────────────
VIDEO_DIR   = r'D:\EndoTracker\selected_dataset'
ANN_DIR     = r'D:\EndoTracker\vis_output\annotation'
OUTPUT_DIR  = r'D:\EndoTracker\output_cotracker'
MAX_FRAMES  = 100
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = torch.hub.load("facebookresearch/co-tracker", "cotracker2").to(device)
model.eval()

COLORS = [(0,0,255),(0,255,0),(255,0,0),(0,255,255),(255,0,255)]

video_files = sorted(Path(VIDEO_DIR).glob("*.mp4"))
print(f"total found {len(video_files)} videos\n")

Using cache found in C:\Users\Administrator/.cache\torch\hub\facebookresearch_co-tracker_main


✅ CoTracker2 加载完成
共找到 5 个视频



In [ ]:

all_results = {}

for video_path in video_files:
    seq_name = video_path.stem
    ann_path = Path(ANN_DIR) / f"annotation_{seq_name}.json"

    if not ann_path.exists():
        print(f"Skip {seq_name}，no annotation file")
        continue

    print(f"{'='*50}\nProcessing: {seq_name}")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  error opening video"); continue
    frames = []
    while len(frames) < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    H, W = frames[0].shape[:2]
    T = len(frames)
    print(f"  Frame={T}, Resolution={H}x{W}")

    video_tensor = torch.tensor(np.array(frames)).permute(0,3,1,2).unsqueeze(0).float().to(device)

    with open(ann_path) as f:
        ann = json.load(f)
    f0_pts  = np.array(ann["f0"])
    gt_last = np.array(ann["fLast"])
    N = len(f0_pts)


    queries = torch.tensor(
        [[0.0, x, y] for x, y in f0_pts],
        dtype=torch.float32
    ).unsqueeze(0).to(device)  # [1, N, 3]

    with torch.no_grad():
        pred_tracks, pred_vis = model(video_tensor, queries=queries)
  
    all_trajs = pred_tracks[0].cpu().numpy()   # [T, N, 2]
    all_vis   = pred_vis[0].cpu().numpy()       # [T, N]
    pred_last = all_trajs[-1]                   # [N, 2]
    errors    = np.linalg.norm(pred_last - gt_last, axis=1)
    ATE       = errors.mean()
    for i in range(N):
        print(f"  P{i+1}: Error={errors[i]:.1f}px  vis={all_vis[-1,i]:.2f}")
    print(f"  Averge ATE: {ATE:.1f}px")



    vis = Visualizer(
        save_dir=OUTPUT_DIR,
        pad_value=120,
        linewidth=2,
        fps=10
    )
    vis.visualize(
        video=video_tensor,
        tracks=pred_tracks,        # [1, T, N, 2]
        visibility=pred_vis,       # [1, T, N]
        filename=f"tracking_{seq_name}"
    )
    print(f"  Tracking video → {OUTPUT_DIR}/tracking_{seq_name}.mp4")

    last_frame = video_tensor[0,-1].permute(1,2,0).cpu().numpy().astype(np.uint8)
    last_frame = cv2.cvtColor(last_frame, cv2.COLOR_RGB2BGR)

    for i in range(N):
        color = COLORS[i % len(COLORS)]
        px = int(round(pred_last[i, 0]))
        py = int(round(pred_last[i, 1]))
        if 0 <= px < W and 0 <= py < H:
            cv2.circle(last_frame, (px,py), 8, color, -1)
            cv2.putText(last_frame, f"P{i+1}", (px+10,py-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        gx = int(round(gt_last[i, 0]))
        gy = int(round(gt_last[i, 1]))
        cv2.drawMarker(last_frame,(gx,gy),(255,255,255),cv2.MARKER_CROSS,20,2)
        cv2.circle(last_frame,(gx,gy),8,(255,255,255),2)
        cv2.putText(last_frame, f"GT{i+1}", (gx+10,gy+15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

        if 0 <= px < W and 0 <= py < H:
            cv2.line(last_frame,(px,py),(gx,gy),(200,200,200),1,cv2.LINE_AA)
            mx, my = (px+gx)//2, (py+gy)//2
            cv2.putText(last_frame, f"{errors[i]:.1f}px", (mx+4,my),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200,200,200), 1)

    cv2.putText(last_frame, "Pred: filled circle", (10,H-50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
    cv2.putText(last_frame, "GT:   white cross+circle", (10,H-30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
    cv2.putText(last_frame, f"ATE={ATE:.1f}px", (10,H-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)

    out_img = str(Path(OUTPUT_DIR) / f"lastframe_{seq_name}.png")
    cv2.imwrite(out_img, last_frame)
    print(f"   Comparison image   → {out_img}")

    # 保存 json
    result = {"seq": seq_name, "ATE": float(ATE),
              "per_point_error": errors.tolist(),
              "pred_last": pred_last.tolist(),
              "gt_last": gt_last.tolist()}
    all_results[seq_name] = result
    with open(Path(OUTPUT_DIR) / f"result_{seq_name}.json", "w") as f:
        json.dump(result, f, indent=2)


print(f"\n{'='*50}\nTotal (CoTracker2):")
ATEs = [r['ATE'] for r in all_results.values()]
for seq, res in all_results.items():
    print(f"  {seq}: ATE={res['ATE']:.1f}px")
print(f"\nTotal Average ATE: {np.mean(ATEs):.1f}px")

with open(Path(OUTPUT_DIR) / "summary.json", "w") as f:
    json.dump({"sequences": all_results, "mean_ATE": float(np.mean(ATEs))}, f, indent=2)
print(f"save to {OUTPUT_DIR}")

处理: stable_seq_000_part_0_dif_0
  帧数=100, 分辨率=480x480
  P1: 误差=7.5px  vis=1.00
  P2: 误差=4.1px  vis=1.00
  P3: 误差=17.6px  vis=1.00
  P4: 误差=34.5px  vis=1.00
  P5: 误差=84.3px  vis=1.00
  平均ATE: 29.6px
Video saved to D:\EndoTracker\output_cotracker\tracking_stable_seq_000_part_0_dif_0.mp4
  ✅ 追踪视频 → D:\EndoTracker\output_cotracker/tracking_stable_seq_000_part_0_dif_0.mp4
  ✅ 对比图   → D:\EndoTracker\output_cotracker\lastframe_stable_seq_000_part_0_dif_0.png
处理: stable_seq_000_part_1_dif_0
  帧数=100, 分辨率=480x480
  P1: 误差=33.4px  vis=1.00
  P2: 误差=120.9px  vis=1.00
  P3: 误差=280.4px  vis=1.00
  P4: 误差=241.0px  vis=1.00
  P5: 误差=46.2px  vis=1.00
  平均ATE: 144.4px
Video saved to D:\EndoTracker\output_cotracker\tracking_stable_seq_000_part_1_dif_0.mp4
  ✅ 追踪视频 → D:\EndoTracker\output_cotracker/tracking_stable_seq_000_part_1_dif_0.mp4
  ✅ 对比图   → D:\EndoTracker\output_cotracker\lastframe_stable_seq_000_part_1_dif_0.png
处理: stable_seq_000_part_2_dif_0
  帧数=100, 分辨率=480x480
  P1: 误差=370.5px  vis=0.00
 